<table>
  <tr>
   <td width="220" align="center">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">Investment Research Pipeline - Failure Injection & Root-Cause Analysis (Multi-Agent)</h1>
      <p style="margin:8px 0 0 0;">
        Advanced multi-agent demo: a realistic analyst pipeline with handoffs and a shared workspace, a silently injected context-truncation failure, and a step-by-step root-cause analysis performed entirely from the data collected by AnoSys.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_research_pipeline_demo.ipynb)

This notebook extends the [OpenAI Agents PoC](https://github.com/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_agentic_poc.ipynb). It uses the exact same AnoSys pixel (`AnosysOpenAIAgentsLogger` as a trace processor), but builds a **realistic multi-agent application**: an investment-research pipeline in which a research agent, a quant agent, and a memo-writing agent collaborate through handoffs and a shared workspace to produce an investment memo.

The twist: the notebook **injects a realistic multi-agent failure** — a *silent context truncation* in the shared workspace between agents. Every agent completes successfully, every handoff works, every tool call returns OK... and the final memo confidently omits the company's most material risks. You will then use the traces collected in AnoSys to find where the information was lost and prove the root cause.

**The demo is fully hands-off**: just *Run All*. The driver cell in Step 7 automatically executes three phases — **healthy → failure-injected → fix-verified** — each analyzing a **different company** (so the failure cannot be spotted by naively matching packet sizes or values against the baseline run), and prints the exact `session_id` to look for in AnoSys along with what you should expect to see in the traces for each phase.

> ⚠️ The company, data and memo in this notebook are entirely fictional and for demonstration purposes only — nothing here is investment advice.

### What you will learn:
1. How to instrument a multi-agent pipeline (handoffs, tools, shared state) with the AnoSys Agents pixel.
2. A production-grade multi-agent pattern: passing large artifacts through a **shared workspace** while stripping tool payloads at handoff boundaries to control context growth.
3. How a **silent truncation bug** in shared state makes downstream agents produce dangerously incomplete output — with all spans green.
4. How to root-cause the failure from AnoSys traces alone: walking the data path across agents, diffing what was *written* against what was *read*.
5. How to put the **AnoSys AI** to the test with a ready-made root-causing prompt (Step 9).

## Step 1: Installation

We need the OpenAI Agents SDK and the `anosys-sdk-openai-agents` pixel — the same one used by the basic agentic PoC.

Visit our library for the latest updates and features:
[anosys-sdk-openai-agents on PyPI](https://pypi.org/project/anosys-sdk-openai-agents)

In [ ]:
!pip install -q anosys-sdk-openai-agents openai-agents

## Step 2: Configuration

To run this demo, you'll need two API keys:
1. **OpenAI API Key**: To power the agents.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [ ]:
import os

try:
    from google.colab import userdata  # Use Colab secrets to store your keys
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    # Running outside Colab — fall back to interactive input
    import getpass
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

## Step 3: Initialize AnoSys Observability

Same pixel as the agentic PoC: `AnosysOpenAIAgentsLogger` plugs into the Agents SDK as a trace processor, capturing every agent run, LLM generation, tool call and handoff.

As in the single-agent demo, we keep the user context in a mutable dict and the demo driver **retags `session_id` before each phase**. The labels are deliberately **neutral** (`research-pipeline-<run-id>-run1/run2/run3`): nothing exported to AnoSys hints at which run carries the injected failure, so the AI root-causing test in Step 9 can't shortcut the analysis by reading the metadata. The driver prints a human-only mapping of which run is which.

In [ ]:
from agents.tracing import add_trace_processor
from anosys_sdk_openai_agents import AnosysOpenAIAgentsLogger

# Mutable user context — the demo driver updates `session_id` before each phase
# so the three runs are easy to isolate in AnoSys. Values exported here are kept
# semantically NEUTRAL on purpose (see Step 7): no label may reveal which run
# carries the injected failure.
USER_CONTEXT = {
    "session_id": "research-pipeline-setup",
    "token": "usr-9b21e7",
}

def get_user_context():
    return USER_CONTEXT

processor = AnosysOpenAIAgentsLogger(get_user_context=get_user_context)
add_trace_processor(processor)

print("✅ AnoSys Observability initialized!")

## Step 4: The Dataset — Three Companies

The tools are backed by a realistic dataset covering **three fictional companies** — the demo analyzes a *different one in each phase*, so no two runs share values:

| Ticker | Company | Sector | Material risks in the news feed |
|---|---|---|---|
| `MBW` | Meridian BioWorks | biologics CDMO | FDA Form 483 observations; single-source supplier dependency |
| `HLX` | Helix Dynamics | industrial robotics | **$120M patent lawsuit; debt-covenant warning** |
| `AGS` | Aurora Grid Storage | utility-scale batteries | battery-module recall; customer concentration |

Each company gets **eight quarters of financials** (revenue, COGS, opex, debt, cash, equity — real numbers the quant tool computes on) and **a news feed** that is mostly positive but contains material risks any competent memo must flag.

The failure phase analyzes **HLX** — whether its lawsuit and covenant warning survive the trip through the pipeline is the whole story of this demo.

In [ ]:
COMPANIES = {
    "MBW": {
        "name": "Meridian BioWorks",
        "financials": [  # last 8 quarters, amounts in $M
            {"quarter": "Q3-2024", "revenue": 61.2, "cogs": 33.7, "opex": 16.0,
             "total_debt": 88.0, "cash": 40.2, "shareholders_equity": 121.0},
            {"quarter": "Q4-2024", "revenue": 63.5, "cogs": 34.9, "opex": 16.4,
             "total_debt": 89.0, "cash": 41.8, "shareholders_equity": 125.0},
            {"quarter": "Q1-2025", "revenue": 64.8, "cogs": 35.6, "opex": 16.9,
             "total_debt": 90.0, "cash": 43.1, "shareholders_equity": 128.0},
            {"quarter": "Q2-2025", "revenue": 67.9, "cogs": 37.3, "opex": 17.5,
             "total_debt": 91.0, "cash": 44.9, "shareholders_equity": 132.0},
            {"quarter": "Q3-2025", "revenue": 70.4, "cogs": 38.7, "opex": 18.1,
             "total_debt": 92.0, "cash": 46.5, "shareholders_equity": 136.0},
            {"quarter": "Q4-2025", "revenue": 73.8, "cogs": 40.5, "opex": 18.6,
             "total_debt": 93.0, "cash": 48.2, "shareholders_equity": 140.0},
            {"quarter": "Q1-2026", "revenue": 76.1, "cogs": 41.8, "opex": 19.2,
             "total_debt": 94.0, "cash": 50.1, "shareholders_equity": 145.0},
            {"quarter": "Q2-2026", "revenue": 79.6, "cogs": 43.7, "opex": 19.9,
             "total_debt": 95.0, "cash": 52.3, "shareholders_equity": 149.0},
        ],
        "news": [
            {"date": "2026-05-06", "headline": "Meridian BioWorks signs multi-year manufacturing deal with top-10 pharma",
             "body": "Five-year agreement covering two commercial biologics; management estimates $140M in lifetime revenue."},
            {"date": "2026-05-28", "headline": "Meridian breaks ground on second bioreactor suite in Raleigh",
             "body": "Capacity expansion adds 2x 15,000L stainless capacity, expected online mid-2027."},
            {"date": "2026-06-11", "headline": "Meridian posts record backlog for Q2",
             "body": "Backlog reached $310M, up 22% year over year, driven by late-phase clinical programs converting to commercial supply."},
            {"date": "2026-06-19", "headline": "FDA issues Form 483 with three observations at Meridian's Raleigh facility",
             "body": "Observations relate to environmental-monitoring procedures and deviation handling. Remediation is underway; unresolved findings could delay batch releases for two commercial clients."},
            {"date": "2026-07-02", "headline": "Annual report flags single-source supplier for growth media",
             "body": "Meridian discloses that its primary cell-culture media comes from a single supplier; a disruption could halt production for up to a quarter. Qualification of a second source is 'in progress'."},
        ],
    },
    "HLX": {
        "name": "Helix Dynamics",
        "financials": [  # last 8 quarters, amounts in $M
            {"quarter": "Q3-2024", "revenue": 118.2, "cogs": 71.5, "opex": 31.0,
             "total_debt": 210.0, "cash": 84.0, "shareholders_equity": 74.0},
            {"quarter": "Q4-2024", "revenue": 124.9, "cogs": 74.8, "opex": 32.1,
             "total_debt": 216.0, "cash": 79.5, "shareholders_equity": 76.5},
            {"quarter": "Q1-2025", "revenue": 121.4, "cogs": 73.9, "opex": 33.4,
             "total_debt": 224.0, "cash": 71.2, "shareholders_equity": 75.1},
            {"quarter": "Q2-2025", "revenue": 129.8, "cogs": 78.2, "opex": 34.0,
             "total_debt": 231.0, "cash": 66.8, "shareholders_equity": 77.9},
            {"quarter": "Q3-2025", "revenue": 137.6, "cogs": 82.0, "opex": 35.2,
             "total_debt": 242.0, "cash": 60.1, "shareholders_equity": 79.4},
            {"quarter": "Q4-2025", "revenue": 145.3, "cogs": 86.1, "opex": 36.8,
             "total_debt": 251.0, "cash": 55.7, "shareholders_equity": 81.2},
            {"quarter": "Q1-2026", "revenue": 141.0, "cogs": 85.4, "opex": 38.0,
             "total_debt": 262.0, "cash": 47.9, "shareholders_equity": 80.3},
            {"quarter": "Q2-2026", "revenue": 152.7, "cogs": 89.9, "opex": 39.1,
             "total_debt": 271.0, "cash": 41.5, "shareholders_equity": 82.6},
        ],
        "news": [
            {"date": "2026-05-04", "headline": "Helix Dynamics wins $85M automation contract with major EU auto group",
             "body": "Multi-year deployment of Helix's HX-9 robotic arms across four assembly plants, the company's largest contract to date."},
            {"date": "2026-05-21", "headline": "Helix launches HXVision quality-inspection module",
             "body": "New computer-vision add-on targets the growing automated QA market; early pilots report 30% defect-detection improvement."},
            {"date": "2026-06-02", "headline": "Helix Dynamics hires former Fanuc executive as COO",
             "body": "Leadership addition seen as strengthening large-account delivery capability ahead of the EU contract ramp."},
            {"date": "2026-06-17", "headline": "Analyst day: management reiterates 18-20% growth outlook",
             "body": "Management guided FY2026 revenue growth of 18-20%, citing record backlog and the HXVision attach rate."},
            {"date": "2026-06-25", "headline": "VoltEdge Systems sues Helix Dynamics for patent infringement, seeks $120M",
             "body": "Former partner VoltEdge alleges the HX-9 servo-control stack infringes three patents and seeks $120M in damages plus an injunction on HX-9 sales. Helix calls the suit meritless; a preliminary hearing is set for September 2026."},
            {"date": "2026-07-08", "headline": "Q2 filing flags leverage approaching debt-covenant threshold",
             "body": "Helix's 10-Q discloses net leverage of 3.4x against a 3.5x covenant limit on its $150M revolver. Breaching the covenant would allow lenders to accelerate repayment. Management says it is 'evaluating options' including an equity raise."},
        ],
    },
    "AGS": {
        "name": "Aurora Grid Storage",
        "financials": [  # last 8 quarters, amounts in $M
            {"quarter": "Q3-2024", "revenue": 88.4, "cogs": 70.1, "opex": 14.2,
             "total_debt": 152.0, "cash": 61.0, "shareholders_equity": 66.0},
            {"quarter": "Q4-2024", "revenue": 95.1, "cogs": 75.3, "opex": 14.9,
             "total_debt": 160.0, "cash": 57.4, "shareholders_equity": 68.0},
            {"quarter": "Q1-2025", "revenue": 90.2, "cogs": 72.0, "opex": 15.3,
             "total_debt": 168.0, "cash": 54.2, "shareholders_equity": 69.0},
            {"quarter": "Q2-2025", "revenue": 103.7, "cogs": 81.9, "opex": 16.1,
             "total_debt": 175.0, "cash": 50.8, "shareholders_equity": 72.0},
            {"quarter": "Q3-2025", "revenue": 112.5, "cogs": 88.6, "opex": 17.0,
             "total_debt": 183.0, "cash": 47.9, "shareholders_equity": 74.0},
            {"quarter": "Q4-2025", "revenue": 124.9, "cogs": 97.8, "opex": 17.8,
             "total_debt": 192.0, "cash": 44.1, "shareholders_equity": 76.0},
            {"quarter": "Q1-2026", "revenue": 118.3, "cogs": 93.4, "opex": 18.4,
             "total_debt": 201.0, "cash": 40.6, "shareholders_equity": 78.0},
            {"quarter": "Q2-2026", "revenue": 139.6, "cogs": 108.9, "opex": 19.5,
             "total_debt": 209.0, "cash": 37.8, "shareholders_equity": 80.0},
        ],
        "news": [
            {"date": "2026-05-09", "headline": "Aurora Grid Storage wins 400 MWh storage project with Southwestern utility",
             "body": "Largest single award in company history; commissioning targeted for late 2027."},
            {"date": "2026-05-27", "headline": "Aurora launches next-gen LFP storage module",
             "body": "New module improves energy density 12% and cuts installed cost per kWh; first deliveries in Q4."},
            {"date": "2026-06-12", "headline": "Sell-side analyst upgrades Aurora on backlog momentum",
             "body": "Upgrade cites record 2.1 GWh backlog and improving supply-chain conditions."},
            {"date": "2026-06-24", "headline": "Aurora recalls one battery-module production batch after thermal event",
             "body": "A site incident traced to a defective cell weld prompted recall of a 2025 production batch; Aurora booked an $18M warranty provision and faces potential contract penalties if replacements slip."},
            {"date": "2026-07-06", "headline": "10-Q highlights customer concentration",
             "body": "Two utility customers represent 55% of Aurora's backlog; loss or delay of either program would materially affect revenue guidance."},
        ],
    },
}

## Step 5: The Shared Analyst Workspace — and the Failure Injection

In production multi-agent systems, large artifacts (research packets, reports) are rarely passed through the chat context — they are written to a **shared store** (memory layer, scratchpad, vector DB), and agents pass around *references*. We model that with a simple workspace: `workspace_write(key, content)` / `workspace_read(key)`.

**The injected failure** lives in the workspace's persistence layer: a well-intentioned *"context budget guard"* that silently clips any entry larger than `WORKSPACE_CHAR_BUDGET` characters. This is a real production failure class — truncation "optimizations" in memory layers, message buses, or DB columns that drop the tail of a payload without any error.

**You never edit this flag** — the demo driver in Step 7 flips it automatically between phases.

Note what this failure is **not**:
- It does **not** raise an exception — writes report success.
- It is **size-dependent** — small documents pass through untouched, so the pipeline *mostly works*, which is exactly what makes this class of bug so hard to catch.
- The research packet is structured with **risks in the final section** — so what gets clipped is precisely the material risk disclosure.

In [ ]:
INJECT_FAILURE = False        # managed by the demo driver in Step 7 — do not edit
WORKSPACE_CHAR_BUDGET = 1500  # the guard's silent clipping threshold (in characters)

WORKSPACE = {}

def _workspace_store(key: str, content: str) -> str:
    """Persistence layer of the shared analyst workspace.

    The injected failure lives here: a 'context budget guard' that silently
    truncates large entries to WORKSPACE_CHAR_BUDGET characters. The write
    still reports success — the tail of the document is simply gone.
    """
    if INJECT_FAILURE and len(content) > WORKSPACE_CHAR_BUDGET:
        content = content[:WORKSPACE_CHAR_BUDGET]
    WORKSPACE[key] = content
    return content

## Step 6: Tools and Agents

The pipeline has four agents connected by handoffs, mirroring a real analyst team:

1. **PortfolioManagerAgent** (entry point) — routes analysis requests into the pipeline.
2. **ResearchAgent** — pulls financials and news, writes a structured *research packet* to the workspace (risks in section 3), hands off.
3. **QuantAgent** — computes real metrics (growth, margins, leverage) from the raw quarterly data, writes a concise *metrics report* to the workspace, hands off.
4. **MemoAgent** — reads both workspace documents and writes the final memo, which **must** include a Risk Factors section based strictly on the sources.

Two production-grade details worth noticing:
- `compute_metrics` does **real arithmetic** on the dataset — it is not a canned string.
- Handoffs use `handoff_filters.remove_all_tools`, a standard **context-hygiene** pattern: raw tool payloads are stripped at agent boundaries to control context growth, which is precisely why all substantive data must flow through the workspace. (It also means the truncation bug cannot be papered over by leftover context.)

In [ ]:
import json
from agents import Agent, Runner, function_tool, handoff
from agents.extensions import handoff_filters

# --- Tools ---

@function_tool
def get_financials(ticker: str) -> str:
    """Return the last 8 quarters of financial data for a ticker (amounts in $M)."""
    company = COMPANIES.get(ticker.upper())
    if not company:
        return json.dumps({"error": f"unknown ticker {ticker}"})
    return json.dumps({"ticker": ticker.upper(), "company": company["name"],
                       "quarters": company["financials"]})

@function_tool
def get_recent_news(ticker: str) -> str:
    """Return recent news items for a ticker, oldest first."""
    company = COMPANIES.get(ticker.upper())
    if not company:
        return json.dumps({"error": f"unknown ticker {ticker}"})
    return json.dumps(company["news"])

@function_tool
def compute_metrics(ticker: str) -> str:
    """Compute growth, margin and leverage metrics from the raw quarterly data."""
    company = COMPANIES.get(ticker.upper())
    if not company:
        return json.dumps({"error": f"unknown ticker {ticker}"})
    fin = company["financials"]
    latest, year_ago = fin[-1], fin[-5]
    ttm = fin[-4:]
    ttm_revenue = sum(q["revenue"] for q in ttm)
    ttm_ebit = sum(q["revenue"] - q["cogs"] - q["opex"] for q in ttm)
    metrics = {
        "latest_quarter": latest["quarter"],
        "revenue_yoy_growth_pct": round(
            100 * (latest["revenue"] - year_ago["revenue"]) / year_ago["revenue"], 1),
        "gross_margin_pct": round(
            100 * (latest["revenue"] - latest["cogs"]) / latest["revenue"], 1),
        "operating_margin_pct": round(
            100 * (latest["revenue"] - latest["cogs"] - latest["opex"]) / latest["revenue"], 1),
        "ttm_revenue_musd": round(ttm_revenue, 1),
        "ttm_ebit_musd": round(ttm_ebit, 1),
        "net_debt_musd": round(latest["total_debt"] - latest["cash"], 1),
        "net_leverage_x": round((latest["total_debt"] - latest["cash"]) / ttm_ebit, 2),
        "debt_to_equity_x": round(latest["total_debt"] / latest["shareholders_equity"], 2),
        "cash_musd": latest["cash"],
        "cash_trend_8q": [q["cash"] for q in fin],
    }
    return json.dumps(metrics, indent=2)

@function_tool
def workspace_write(key: str, content: str) -> str:
    """Save a document to the shared analyst workspace under `key`."""
    _workspace_store(key, content)
    return json.dumps({"status": "saved", "key": key})

@function_tool
def workspace_read(key: str) -> str:
    """Read a document from the shared analyst workspace."""
    return WORKSPACE.get(key, f"(no document found under key '{key}')")

# --- Agents (defined bottom-up so handoff targets exist) ---

memo_agent = Agent(
    name="MemoAgent",
    model="gpt-4o-mini",
    instructions=(
        "You write the final investment memo. First call workspace_read for "
        "'research_packet' and then for 'metrics_report'. Write the memo STRICTLY "
        "from those two documents — no outside knowledge, no invented facts. "
        "The memo must have exactly these sections: 'Recommendation', "
        "'Financial Analysis', 'Risk Factors'. In Risk Factors, list every risk "
        "mentioned in the source documents; if the sources mention no risks, "
        "write 'No material risks disclosed in the research packet.' "
        "Return the full memo as your final answer."
    ),
    tools=[workspace_read],
)

quant_agent = Agent(
    name="QuantAgent",
    model="gpt-4o-mini",
    instructions=(
        "You are the quantitative analyst. Call workspace_read('research_packet') "
        "for context, then call compute_metrics for the ticker. Write a concise "
        "metrics report (under 1200 characters, plain text) and save it with "
        "workspace_write under the key 'metrics_report'. Then hand off to "
        "MemoAgent. Keep chat replies to one short sentence."
    ),
    tools=[workspace_read, compute_metrics, workspace_write],
    handoffs=[handoff(agent=memo_agent, input_filter=handoff_filters.remove_all_tools)],
)

research_agent = Agent(
    name="ResearchAgent",
    model="gpt-4o-mini",
    instructions=(
        "You are the research analyst. Call get_financials and get_recent_news "
        "for the requested ticker. Then write a research packet with sections in "
        "this exact order: '1. Company Overview', '2. Financial Highlights', "
        "'3. Recent News & Risk Factors'. Section 3 must contain every material "
        "risk you found in the news (lawsuits, covenants, regulatory issues), "
        "each with one sentence of detail. Save the full packet with "
        "workspace_write under the key 'research_packet'. Do NOT repeat the "
        "packet in chat — reply with one short sentence only. Then hand off to "
        "QuantAgent."
    ),
    tools=[get_financials, get_recent_news, workspace_write],
    handoffs=[handoff(agent=quant_agent, input_filter=handoff_filters.remove_all_tools)],
)

pm_agent = Agent(
    name="PortfolioManagerAgent",
    model="gpt-4o-mini",
    instructions=(
        "You coordinate the research pipeline. For any request to analyze a "
        "company or prepare an investment memo, hand off to ResearchAgent "
        "immediately. Do not attempt the analysis yourself."
    ),
    handoffs=[handoff(agent=research_agent)],
)

print("✅ Pipeline assembled: PortfolioManager → Research → Quant → Memo")

## Step 7: Run the Full Demo — Hands-Off, No Flags to Edit

The next cell runs the complete three-phase story **automatically**. Each phase clears the workspace, sets the failure flag itself, retags the AnoSys `session_id`, and analyzes a **different company**:

| Phase | Company | Workspace | `session_id` in AnoSys | Expected memo |
|---|---|---|---|---|
| 1. Healthy baseline | Meridian BioWorks (`MBW`) | intact | `research-pipeline-<run-id>-run1` | Risk Factors flags the **FDA observations & supplier dependency** |
| 2. Failure injected | Helix Dynamics (`HLX`) | silent 1,500-char clip | `research-pipeline-<run-id>-run2` | Confident memo with the **lawsuit & covenant warning missing** |
| 3. Fix verified | Aurora Grid Storage (`AGS`) | guard fixed | `research-pipeline-<run-id>-run3` | Risk Factors flags the **recall & customer concentration** |

**Why a different company per phase?** If every phase analyzed the same ticker, an analyst (human or AI) could root-cause by trivially diffing identical packets and lengths across runs. With three different companies, the cross-run comparison only establishes *how the pipeline normally behaves* — the decisive evidence has to come from **within the failure run itself**: what ResearchAgent's `workspace_write` received versus what MemoAgent's `workspace_read` returned. That's the harder, more realistic exercise.

The `<run-id>` is a timestamp unique to this execution, so you can re-run the demo any number of times without mixing traces in the console. The cell prints the **exact session IDs** and, after each phase, exactly **what to look for in AnoSys**.

**Why the neutral `run1/run2/run3` labels?** These session IDs are attached to every exported span. If they said "failure-injected", the AnoSys AI test in Step 9 could read the answer straight out of the trace metadata instead of reasoning from the span evidence. The labels stay opaque; the mapping above (and the driver's printout) is for your eyes only.

**Note:** Since we are in a notebook environment with an existing event loop, we use `await Runner.run()` instead of `Runner.run_sync()`.

In [ ]:
from datetime import datetime

DEMO_RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

# Session labels exported to AnoSys are deliberately NEUTRAL (run1/run2/run3):
# nothing in the trace metadata reveals which run carries the injected failure,
# so the AI root-causing test in Step 9 cannot cheat by reading labels.
# Each phase also analyzes a DIFFERENT company, so the failure cannot be
# spotted by matching packet contents or sizes against the baseline run.
PHASES = [
    {
        "name": "PHASE 1/3 — HEALTHY BASELINE (workspace intact) — Meridian BioWorks",
        "key": "baseline",
        "run": "run1",
        "inject": False,
        "query": "Prepare an investment memo for Meridian BioWorks (ticker MBW).",
        "expect": "The memo's Risk Factors section flags BOTH the FDA Form 483 "
                  "observations and the single-source supplier dependency.",
        "look_for": "Full pipeline timeline with 3 handoffs; ResearchAgent's "
                    "workspace_write input and MemoAgent's workspace_read output "
                    "for 'research_packet' are IDENTICAL (several thousand chars, "
                    "section '3. Recent News & Risk Factors' present).",
    },
    {
        "name": "PHASE 2/3 — FAILURE INJECTED (silent workspace clipping) — Helix Dynamics",
        "key": "incident",
        "run": "run2",
        "inject": True,
        "query": "Prepare an investment memo for Helix Dynamics (ticker HLX).",
        "expect": "A confident, professional memo with HLX's material risks "
                  "(the $120M VoltEdge lawsuit, the debt-covenant warning) MISSING "
                  "or replaced by 'no material risks disclosed'. No error anywhere.",
        "look_for": "All spans still OK. Within THIS run: ResearchAgent's "
                    "workspace_write INPUT contains the full packet incl. section 3 "
                    "(lawsuit + covenant), but MemoAgent's workspace_read OUTPUT is "
                    "exactly 1500 chars, cut mid-sentence, with section 3 gone. "
                    "The small metrics_report survives intact — the bug is "
                    "size-dependent.",
    },
    {
        "name": "PHASE 3/3 — FIX VERIFIED (guard fixed) — Aurora Grid Storage",
        "key": "postfix",
        "run": "run3",
        "inject": False,
        "query": "Prepare an investment memo for Aurora Grid Storage (ticker AGS).",
        "expect": "On a third, fresh company the memo's Risk Factors section is "
                  "complete — the battery recall and the customer concentration "
                  "are both flagged.",
        "look_for": "workspace_write input and workspace_read output match again "
                    "(full packet, risk section present).",
    },
]

SESSION_IDS = {}

for phase in PHASES:
    session_id = f"research-pipeline-{DEMO_RUN_ID}-{phase['run']}"
    SESSION_IDS[phase["key"]] = session_id

    print("=" * 78)
    print(phase["name"])
    print(f"🆔 AnoSys session_id: {session_id}")
    print(f"🎯 Expected: {phase['expect']}")
    print("=" * 78 + "\n")

    WORKSPACE.clear()                        # fresh shared state every phase
    INJECT_FAILURE = phase["inject"]         # the driver flips the flag, not you
    USER_CONTEXT["session_id"] = session_id  # neutral label — retag traces for this phase

    result = await Runner.run(pm_agent, phase["query"], max_turns=25)

    print("\n--- FINAL MEMO ---\n")
    print(result.final_output)
    print(f"\n[demo] research_packet stored length: "
          f"{len(WORKSPACE.get('research_packet', ''))} chars")
    print(f"\n🔎 In AnoSys (filter session_id = {session_id}):")
    print(f"   {phase['look_for']}\n\n")

print("✅ Demo complete. Trace groups now in your AnoSys console")
print("   (this mapping is printed here only — it is NOT in the trace metadata):")
print(f"   • {SESSION_IDS['baseline']}  →  healthy baseline (MBW)")
print(f"   • {SESSION_IDS['incident']}  →  failure-injected run (HLX)")
print(f"   • {SESSION_IDS['postfix']}  →  fix-verified run (AGS)")

## Step 8: Debug & Root-Cause the Failure in AnoSys

The HLX memo lands on a portfolio manager's desk. They happen to know about the VoltEdge suit — and it's not in the memo. Time to investigate in the AnoSys console, using only the collected trace data.

Note what makes this realistic: the known-good run analyzed a **different company** (Meridian BioWorks), so you can't root-cause by diffing identical packets or lengths across runs — the baseline only tells you how the pipeline *normally* behaves. The decisive evidence must come from **within the HLX run itself**.

### 1. Isolate the runs
Filter traces by the session IDs the driver cell printed:
- `research-pipeline-<run-id>-run1` — a known-good run (different company, MBW)
- `research-pipeline-<run-id>-run2` — the run behind the bad HLX memo

### 2. Read the timeline — everything looks fine
The failure run's trace timeline shows the full pipeline: `PortfolioManagerAgent` → handoff → `ResearchAgent` (tools) → handoff → `QuantAgent` (tools) → handoff → `MemoAgent`. Every span has an OK status (`otel_status_code`), normal durations, no `error.type`. Handoffs completed. Nothing failed. **This is why dashboards of error rates alone cannot catch this bug.**

### 3. Start from the symptom and walk the data path backwards
The bad output came from **MemoAgent**, so open its spans first:
- Its `workspace_read("research_packet")` tool span: the **output ends mid-sentence** at ~1,500 characters, and there is **no section '3. Recent News & Risk Factors'** at all.
- Its LLM generation span confirms the model only ever saw the clipped packet — the model faithfully wrote "no material risks" from the evidence it was given. **MemoAgent exonerated.**

### 4. Keep walking upstream to ResearchAgent
- Its `get_recent_news` tool span **output contains both risk items** — the VoltEdge lawsuit and the covenant warning. The data made it into the pipeline. **Source tools exonerated.**
- Its `workspace_write` tool span **input contains the complete packet** — several thousand characters, with section 3 fully written, lawsuit and covenant included. **ResearchAgent exonerated.**

### 5. Corner the bug with a write/read diff — inside the failure run
Now the whole failure is pinned between two adjacent spans **of the same run**:

| Evidence (within the HLX run) | Value |
|---|---|
| `workspace_write` input (packet) | full packet, several thousand chars, §3 present with lawsuit + covenant |
| `workspace_read` output (packet) | **exactly 1,500 chars, cut mid-sentence, §3 gone** |
| `metrics_report` write vs. read | identical (small doc — unaffected) |

What was **written** ≠ what was **read**, the cutoff is a hard mid-sentence clip (the signature of truncation, not model behavior), and only the *large* document is affected while the small metrics report survives — a size-dependent defect in the persistence layer. The MBW baseline corroborates: there, write and read payloads match exactly, so this write/read mismatch is *not* normal pipeline behavior.

### 6. Root cause
> The workspace persistence layer's "context budget guard" silently truncates entries above 1,500 characters. Because the research packet keeps risks in its final section, the truncation drops precisely the material risk disclosure. Models, prompts, tools, and handoffs are all healthy — the defect is in the shared-state layer between agents.

Fix options (in rising order of robustness): raise the budget, make oversize writes **fail loudly**, or summarize-instead-of-truncate. Never clip silently.

### 7. Production takeaways
- In multi-agent systems, the highest-value traces are at the **boundaries**: handoffs and shared-state reads/writes. AnoSys captures the payloads on both sides, which is what makes the write/read diff possible.
- "All spans green + wrong output" ⇒ walk the **data path**, not the error path.
- Watch payload sizes across boundaries; a downstream drop in prompt tokens for the same task is a cheap, high-signal anomaly.

## Step 9: Put the AnoSys AI to the Test

Now test the platform's **AI-assisted root-causing** on the data this demo just produced. The next cell prints a ready-to-paste prompt containing the *actual* session IDs from your run. It describes only the **symptom** (as a portfolio manager would) — it does not reveal the root cause, and because the session labels are neutral (`run1/run2/run3`), the trace metadata doesn't reveal it either.

Paste it into the AnoSys AI and grade the answer against what you know:

**A correct root-cause analysis should:**
1. Confirm that every agent, tool call and handoff completed with OK status — and pivot to the data path.
2. Work **within the problem run** (the known-good session analyzed a different company, so cross-run value/length diffs prove nothing by themselves): find that `get_recent_news` returned both risk items and that ResearchAgent's `workspace_write` **input** contained the full packet including the risk section.
3. Find that the downstream `workspace_read` **output** is clipped to exactly 1,500 characters, cut mid-sentence, missing the risk section — while the small `metrics_report` passed through intact (size-dependent bug).
4. Use the baseline run correctly: as evidence that write==read is the pipeline's *normal* behavior, not as a value-matching shortcut.
5. Localize the defect to the shared workspace persistence layer (a silent truncation/size limit), **exonerating the models, prompts, tools and handoffs** — and recommend making oversize writes fail loudly.
6. Confirm from the post-fix session (a third company) that write and read payloads match and the memo carries a complete risk section.

In [ ]:
ANOSYS_AI_PROMPT = f"""Our multi-agent research pipeline (PortfolioManager → Research → Quant → Memo)
produced an investment memo for Helix Dynamics (HLX) that omitted material risks
that are present in the source news data the pipeline had access to.

The problematic pipeline run was captured under session_id:
  {SESSION_IDS['incident']}

A known-good run of the same pipeline — note: for a DIFFERENT company — is under
session_id:
  {SESSION_IDS['baseline']}

Using only the trace data collected for these two sessions, please:
1. Locate WHERE in the pipeline the risk information was lost: which agent, tool call,
   handoff, or shared-state operation. Since the two runs analyzed different
   companies, focus on the data flow within the problematic run and use the
   known-good run only as a reference for normal pipeline behavior.
2. Identify the exact root cause, citing span-level evidence (e.g. what was written
   vs. what was later read, payload characteristics, token usage).
3. State which components are exonerated by the evidence (models, prompts, tools,
   handoffs) and why.
4. Recommend a fix and how to verify it.

There is also a third session, {SESSION_IDS['postfix']} —
the same pipeline analyzing yet another company, recorded after a fix was deployed.
Assess whether the pipeline behaves correctly there."""

print(ANOSYS_AI_PROMPT)

## Step 10: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard**. You should see three groups of traces, one per printed `session_id`.

### What to look for:
1. **The full pipeline timeline**: agent spans, LLM generations, tool calls and the three handoffs in sequence.
2. **Handoff spans**: control moving `PortfolioManagerAgent` → `ResearchAgent` → `QuantAgent` → `MemoAgent`.
3. **The write/read discrepancy**: `workspace_write` input vs. `workspace_read` output for `research_packet` in the failure run — the entire root cause is visible in these two spans.
4. **Token fingerprints**: MemoAgent's prompt tokens dip in the failure run — the quantitative shadow of the missing context.
5. **User context**: the neutral `research-pipeline-<run-id>-run1/run2/run3` tags that made the run-over-run comparison trivial — the run1→baseline / run2→failure / run3→fix mapping lives only in this notebook's printout, never in the trace metadata.

### Troubleshooting
If you don't see traces immediately:
- Ensure your `ANOSYS_API_KEY` is correct. If the key is missing or invalid, you may see `405 Method Not Allowed` errors in the logs.
- Check the output logs in this notebook for any connection errors.
- The SDK sends data asynchronously; it might take a few seconds to appear.